# ELIZA — Das DOCTOR-Script (Deutsche Version)

**Seminar: Mapping the Machine — Writing With AI as A Critical Practice**  
**Doppelsitzung 4: But Do They Really Care About Us?**

---

## Hintergrund

ELIZA wurde 1964–66 von Joseph Weizenbaum am MIT entwickelt. Das bekannteste Script — DOCTOR — simuliert eine Gesprächsführung nach dem Vorbild der klientenzentrierten Psychotherapie (Carl Rogers).

**Wichtig:** ELIZA *versteht* nichts. Das System arbeitet ausschließlich mit **Pattern-Matching** (Mustererkennung durch reguläre Ausdrücke) und **Substitution** (Austausch von Pronomen). Es gibt kein Weltwissen, keine Semantik, keine Intentionalität.

Dennoch entwickelten Nutzer:innen emotionale Bindungen an das System — ein Phänomen, das als **ELIZA-Effekt** bekannt wurde: die Tendenz, einfache Outputs als echtes Verständnis zu deuten.

### Wie ELIZA funktioniert

Der Ablauf jeder Antwort folgt drei Schritten:

1. **Keyword-Erkennung:** Die Eingabe wird nach Schlüsselwörtern durchsucht (z. B. *Mutter*, *traurig*, *wenn*). Jedes Keyword hat eine Priorität.
2. **Decomposition:** Ein regulärer Ausdruck zerlegt den Satz und extrahiert relevante Fragmente (z. B. aus *Ich fühle mich einsam* wird *einsam* extrahiert).
3. **Reassembly:** Das Fragment wird in eine Antwort-Vorlage eingesetzt (z. B. *Seit wann fühlst du dich einsam?*).

Dazwischen findet eine **Pronomen-Reflexion** statt: *ich* wird zu *du*, *mein* zu *dein* usw. — damit die Rückfrage grammatisch funktioniert.

### Übungen in diesem Notebook

- **Übung A:** Mit ELIZA chatten und den ELIZA-Effekt beobachten
- **Übung B:** Die Persönlichkeit von ELIZA verändern — gleiche Engine, andere Persona


---

## Teil 1: Pronomen-Reflexion

Die Reflexionstabelle ist das Herzstück des Tricks: Sie tauscht die Perspektive, sodass ELIZAs Rückfragen grammatisch korrekt wirken.

Wenn jemand sagt *Ich bin traurig*, muss ELIZA antworten können: *Warum bist du traurig?*

Das ist reine Zeichenkettensubstitution — kein Verständnis.


In [ ]:
import re
import random

# ============================================================
# REFLEXIONEN (Pronomen-Tausch: Ich <-> Du)
# ============================================================
# Diese Tabelle bildet den Perspektivwechsel ab.
# Wenn die Nutzerin sagt "ich bin muede", muss ELIZA
# in der Rueckfrage "du bist muede" verwenden koennen.
#
# UEBUNG B: Hier koenntet ihr z.B. eine Sie-Form einbauen,
# wenn ELIZA formeller werden soll.
# ============================================================

REFLECTIONS = {
    # Pronomen & Possessiva
    "ich": "du", "mich": "dich", "mir": "dir",
    "mein": "dein", "meine": "deine", "meinen": "deinen",
    "meinem": "deinem", "meiner": "deiner",
    "du": "ich", "dich": "mich", "dir": "mir",
    "dein": "mein", "deine": "meine", "deinen": "meinen",
    "deinem": "meinem", "deiner": "meiner",
    "wir": "ihr", "uns": "euch", "unser": "euer", "unsere": "eure",
    "ihr": "wir", "euch": "uns", "euer": "unser", "eure": "unsere",

    # Verben (sein/haben/koennen)
    "bin": "bist", "bist": "bin",
    "war": "warst", "warst": "war",
    "werde": "wirst", "wirst": "werde",
    "habe": "hast", "hast": "habe",
    "kann": "kannst", "kannst": "kann",

    # Sonstiges
    "meinung": "deine meinung",
    "deine meinung": "meine meinung",
}

def reflect(fragment: str) -> str:
    tokens = fragment.lower().split()
    return " ".join(REFLECTIONS.get(w, w) for w in tokens)


### Reflexion ausprobieren

Testet, wie die Pronomen-Reflexion funktioniert:


In [ ]:
# Testet die Reflexion mit eigenen Saetzen:
print(reflect("ich bin traurig"))
print(reflect("meine Mutter versteht mich nicht"))
print(reflect("du kannst mir nicht helfen"))


---

## Teil 2: Keyword-Erkennung

ELIZA durchsucht die Eingabe nach **Schlüsselwörtern** — geordnet nach Priorität. Emotionale und thematische Begriffe (Angst, Trauer, Stress) werden zuerst geprüft, allgemeine Wörter (*ich*, *du*) zuletzt.

Jedes Keyword ist mit einem **regulären Ausdruck** (Regex) verknüpft, der verschiedene Schreibweisen und Flexionsformen abfängt.

**Beachtet:** Die Reihenfolge in `KEYWORD_ORDER` (weiter unten) bestimmt, welches Keyword gewinnt, wenn mehrere in einem Satz vorkommen. Das ist eine zentrale Designentscheidung — sie bestimmt, worüber ELIZA *lieber* sprechen möchte.


In [ ]:
# ============================================================
# KEYWORD-ERKENNUNG (Trigger-Woerter)
# ============================================================
# Jeder Eintrag verbindet einen internen Keyword-Namen
# mit einem regulaeren Ausdruck, der verschiedene
# Schreibweisen und Flexionsformen abfaengt.
#
# UEBUNG B: Hier koennt ihr neue Keywords hinzufuegen,
# um ELIZA auf andere Themen reagieren zu lassen.
# ============================================================

KEYWORD_PATTERNS = {
    "entschuldigung": r"\b(entschuldige|entschuldigung|sorry|verzeih(ung)?)\b",
    "erinnern":       r"\b(erinner(e|st|t|n)?|erinnerung(en)?)\b",
    "wenn":           r"\bwenn\b",
    "traum":          r"\b(träum(e|st|t|en)?|traum|träume|geträumt)\b",
    "vielleicht":     r"\bvielleicht\b",
    "hallo":          r"\b(hallo|hi|hey|servus|moin|guten (tag|morgen|abend))\b",
    "computer":       r"\b(computer|rechner|technik|ki|künstliche intelligenz)\b",
    "name":           r"\bname\b",
    "ja":             r"\b(ja|jep|jo|genau)\b",
    "nein":           r"\b(nein|nee|nö|nope)\b",
    "familie":        r"\b(mutter|vater|mama|papa|schwester|bruder|familie|eltern|kind|kinder)\b",
    "fuehlen":        r"\b(ich (fühle|fuehle)|ich (bin|sei) .*\b(traurig|glücklich|wütend|ärgerlich|ängstlich|aengstlich|gestresst)\b|traurig|glücklich|wütend|ärgerlich|ängstlich|aengstlich|gestresst|depressiv|niedergeschlagen)\b",
    "ich_bin":        r"\bich bin\b",
    "du_bist":        r"\bdu bist\b",
    "kann":           r"\b(kannst du|kann ich)\b",
    "warum":          r"\bwarum\b",
    "weil":           r"\bweil\b",
    "will":           r"\bich (will|möchte)\b",
    "brauche":        r"\bich brauche\b",
    "mag":            r"\bich (mag|liebe)\b",
    "hasse":          r"\bich (hasse|mag nicht|gefällt mir nicht)\b",
    "du":             r"\bdu\b",
    "ich":            r"\bich\b",
    # thematische Cluster (höhere Priorität)
    "stress":         r"\b(stress|gestresst|überfordert|überwältigt)\b",
    "angst":          r"\b(angst|besorgt|sorge|nervös|panik|furcht)\b",
    "glueck":         r"\b(glücklich|froh|zufrieden|freu(e|st|t)|freude)\b",
    "traurig":        r"\b(traurig|depressiv|down|niedergeschlagen)\b",
}


---

## Teil 3: Regeln (Decomposition & Reassembly)

Das Herzstück von ELIZA: Für jedes Keyword gibt es eine oder mehrere **Regeln**. Jede Regel besteht aus:

1. Einem **Decomposition-Pattern** (regulärer Ausdruck), der den Satz zerlegt und Teile extrahiert
2. Einer Liste von **Reassembly-Templates**, in die die extrahierten Teile eingesetzt werden

**Beispiel:** Wenn jemand sagt *Ich fühle mich einsam*:
- Das Pattern `ich fühle mich (.*)` extrahiert `einsam`
- Das Template `Seit wann fühlst du dich {0}?` wird zu: *Seit wann fühlst du dich einsam?*

Die letzte Regel (`default`) greift immer, wenn kein Keyword erkannt wird — das sind ELIZAs generische Rückfragen wie *Erzähl mir mehr*.

**ÜBUNG B: Hier verändert ihr ELIZAs Persönlichkeit!** Ändert die Antwort-Templates, um eine andere Persona zu erzeugen.


In [ ]:
# ============================================================
# REGELN (Decomposition -> Reassembly)
# ============================================================
# Jeder Eintrag hat die Form:
#   "keyword": [(regex_pattern, [antwort1, antwort2, ...]), ...]
#
# {0}, {1} etc. sind Platzhalter fuer die extrahierten
# Textfragmente (nach Pronomen-Reflexion).
#
# UEBUNG B: Veraendert die Antworten, um ELIZA eine
# andere Tonalitaet zu geben. Moeglichkeiten:
#   - sachlich-buerokratisch
#   - streng-autoritaer
#   - poetisch-assoziativ
#   - genervt / desinteressiert
#   - uebertrieben enthusiastisch
# ============================================================

RULES = {
    "hallo": [
        (r".*", ["Hallo – womit möchtest du beginnen?",
                 "Guten Tag. Was beschäftigt dich heute?"])
    ],
    "entschuldigung": [
        (r".*", ["Bitte entschuldige dich nicht.",
                 "Entschuldigungen sind nicht nötig – erzähl mir stattdessen mehr."])
    ],
    "erinnern": [
        (r".*\berinnerst du dich an\b (.*)",
         ["Denkst du oft an {0}?", "Warum ist {0} für dich gerade wichtig?",
          "Welche Gefühle verbindest du mit {0}?"]),
        (r".*\bich erinnere mich (?:an|daran)\b (.*)",
         ["Was macht die Erinnerung an {0} so präsent?",
          "Erinnert dich {0} an eine bestimmte Situation?",
          "Welche Bedeutung hat {0} für dich?"]),
        (r".*", ["An was möchtest du dich erinnern?"])
    ],
    "wenn": [
        (r".*\bwenn\b (.*)",
         ["Glaubst du, es wäre anders, wenn {0}?",
          "Was wäre, wenn {0}? Welche Folgen hätte das?",
          "Wie wahrscheinlich ist es, dass {0} eintritt?"])
    ],
    "traum": [
        (r".*\bich (?:träume?|träumte|habe geträumt)\b(?: von)? (.*)",
         ["Erzähl mir mehr über deinen Traum von {0}.",
          "Glaubst du, der Traum über {0} hat eine Bedeutung?",
          "Weshalb kommt dir {0} gerade jetzt in den Sinn?"]),
        (r".*\btraum|träume\b(.*)",
         ["Welche Träume beschäftigen dich{0}?",
          "Kommen diese Träume häufig vor{0}?"])
    ],
    "vielleicht": [
        (r".*", ["Bist du dir unsicher?",
                 "Was spricht dafür – und was dagegen?",
                 "Wie könntest du mehr Klarheit gewinnen?"])
    ],
    "computer": [
        (r".*", ["Weshalb erwähnst du Computer?",
                 "Glaubst du, Technik beeinflusst deine Gefühle?",
                 "Welche Rolle spielt KI für dich in diesem Zusammenhang?"])
    ],
    "name": [
        (r".*", ["Namen sind nicht so wichtig – erzähl mir lieber, wie du dich fühlst."])
    ],
    "ja": [
        (r".*", ["Verstehe. Magst du das ausführen?",
                 "Worin stimmst du genau zu?"])
    ],
    "nein": [
        (r".*", ["Was spricht dagegen?",
                 "Was hält dich davon ab?"])
    ],
    "familie": [
        (r".*\b(mutter|vater|mama|papa|schwester|bruder|familie|eltern|kind|kinder)\b(.*)",
         ["Erzähl mir mehr über {0}.",
          "Welche Beziehung hast du zu {0}?",
          "Inwiefern spielt {0} hier eine Rolle?"])
    ],
    "fuehlen": [
        (r".*\bich (?:fühle|fuehle) mich\b (.*)",
         ["Seit wann fühlst du dich {0}?",
          "Welche Situationen rufen {0} hervor?",
          "Wie gehst du mit {0} um?"]),
        (r".*\b(traurig|glücklich|wütend|ärgerlich|ängstlich|aengstlich|gestresst|depressiv|niedergeschlagen)\b(.*)",
         ["Erzähl mir mehr darüber, dass du dich {0} fühlst.",
          "Was löst dieses Gefühl {0} bei dir aus?",
          "Gibt es Strategien, die dir bei {0} helfen?"])
    ],
    "ich_bin": [
        (r".*\bich bin\b (.*)",
         ["Glaubst du, dass du wirklich {0} bist?",
          "Was führt dich zu der Überzeugung, {0} zu sein?",
          "Wärst du gerne anders als {0}?"])
    ],
    "du_bist": [
        (r".*\bdu bist\b (.*)",
         ["Interessant, dass du denkst, ich sei {0}.",
          "Was bedeutet es für dich, wenn ich {0} bin?"])
    ],
    "kann": [
        (r".*\bkannst du\b (.*)",
         ["Glaubst du, ich könnte {0}?",
          "Vielleicht kannst du selbst {0}. Woran würdest du das merken?"]),
        (r".*\bkann ich\b (.*)",
         ["Warum möchtest du {0}?",
          "Was würde passieren, wenn du {0} würdest?"])
    ],
    "warum": [
        (r".*\bwarum\b(.*)",
         ["Was denkst du selbst – warum{0}?",
          "Welche Gründe fallen dir ein{0}?"])
    ],
    "weil": [
        (r".*\bweil\b(.*)",
         ["Ist das der einzige Grund{0}?",
          "Gibt es noch weitere Gründe{0}?"])
    ],
    "will": [
        (r".*\bich (?:will|möchte)\b (.*)",
         ["Warum möchtest du {0}?",
          "Was würde es für dich bedeuten, {0} zu erreichen?",
          "Was hindert dich daran, {0}?"])
    ],
    "brauche": [
        (r".*\bich brauche\b (.*)",
         ["Weshalb brauchst du {0}?",
          "Wie würdest du dich fühlen, wenn du {0} hättest?"])
    ],
    "mag": [
        (r".*\bich (?:mag|liebe)\b (.*)",
         ["Was genau magst du an {0}?",
          "Was macht {0} für dich attraktiv?"])
    ],
    "hasse": [
        (r".*\bich (?:hasse|mag nicht|gefällt mir nicht)\b (.*)",
         ["Was stört dich an {0}?",
          "Seit wann empfindest du das gegenüber {0}?"])
    ],
    "du": [
        (r".*\bdu\b (.*)",
         ["Sprechen wir lieber über dich.",
          "Was sagt {0} über dich aus?"])
    ],
    "ich": [
        (r".*\bich\b (.*)",
         ["Erzähl mir mehr über dich.",
          "Warum sagst du: {0}?"])
    ],
    "stress": [
        (r".*\b(stress|gestresst|überfordert|überwältigt)\b(.*)",
         ["Was genau verursacht diesen Stress für dich?",
          "Wie gehst du normalerweise mit Stress um?",
          "Hat sich dein Stresslevel in letzter Zeit verändert?"])
    ],
    "angst": [
        (r".*\b(angst|besorgt|sorge|nervös|panik|furcht)\b(.*)",
         ["Seit wann fühlst du dich ängstlich?",
          "In welchen Situationen tritt die Angst auf?",
          "Was hilft dir, wenn du Angst spürst?"])
    ],
    "glueck": [
        (r".*\b(glücklich|froh|zufrieden|freu(e|st|t)|freude)\b(.*)",
         ["Schön! Was macht dich gerade glücklich?",
          "Gibt es etwas Bestimmtes, das zu diesem Glück beiträgt?",
          "Wie könntest du mehr davon kultivieren?"])
    ],
    "traurig": [
        (r".*\b(traurig|depressiv|down|niedergeschlagen)\b(.*)",
         ["Es tut mir leid zu hören, dass du dich traurig fühlst. Magst du erzählen, warum?",
          "Seit wann fühlst du dich so?",
          "Gibt es etwas, das dir helfen könnte, dich besser zu fühlen?"])
    ],
    "default": [
        (r".*", ["Bitte, erzähl weiter.",
                 "Verstehe. Kannst du das präzisieren?",
                 "Welche Beispiele fallen dir dazu ein?",
                 "Warum sagst du das gerade jetzt?",
                 "Das ist interessant. Kannst du das näher erläutern?"])
    ]
}


---

## Teil 4: Priorität & Matching-Logik

Die Reihenfolge in `KEYWORD_ORDER` bestimmt, welches Keyword gewinnt, wenn mehrere in einem Satz vorkommen. Das ist eine **Designentscheidung**, die ELIZAs Gesprächsverhalten prägt:

- Emotionale Keywords (`stress`, `angst`, `traurig`) haben Vorrang
- Dann kommen thematische Trigger (`familie`, `traum`, `erinnern`)
- Zuletzt die allgemeinen Pronomen (`ich`, `du`)

**Frage für die Diskussion:** Was passiert, wenn ihr die Prioritäten umkehrt? Wenn ELIZA z. B. immer zuerst auf *du* reagiert statt auf Emotionen?


In [ ]:
# ============================================================
# KEYWORD-PRIORITAET
# ============================================================
# Reihenfolge = Prioritaet: Das erste erkannte Keyword gewinnt.
#
# UEBUNG B: Verschiebt Keywords, um das Gespraechsverhalten
# zu veraendern. Was passiert, wenn "computer" ganz oben steht?
# ============================================================

KEYWORD_ORDER = [
    # thematische/emotionale zuerst
    "stress", "angst", "glueck", "traurig",
    # dialogische Marker
    "entschuldigung", "erinnern", "wenn", "traum", "vielleicht",
    "computer", "familie", "fuehlen", "ich_bin", "du_bist",
    "kann", "warum", "weil", "will", "brauche", "mag", "hasse",
    # allgemein
    "du", "ich", "hallo", "name", "ja", "nein",
    "default"
]


In [ ]:
# ============================================================
# NORMALISIERUNG & MATCHING
# ============================================================
# Diese Funktionen bilden die "Engine" von ELIZA:
# 1. _normalize: Eingabe bereinigen (Kleinschreibung, Sonderzeichen)
# 2. _find_keyword: Erstes passendes Keyword finden
# 3. _match_rule: Passendes Decomposition-Pattern anwenden
# 4. respond: Alles zusammenfuehren -> Antwort generieren
# ============================================================

def _normalize(text: str) -> str:
    # Umlaute/ß behalten, Sonderzeichen (außer .,!?;:-) entfernen, Whitespace normalisieren
    text = text.strip()
    text = re.sub(r"[^\w\s\.,!\?:;\-äöüÄÖÜß]", "", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text, flags=re.UNICODE)
    return text.lower()

def _find_keyword(text: str) -> str:
    for kw in KEYWORD_ORDER:
        if kw == "default":
            continue
        pattern = KEYWORD_PATTERNS.get(kw)
        if pattern and re.search(pattern, text, flags=re.UNICODE):
            return kw
    return "default"

def _match_rule(kw: str, text: str):
    for (pattern, replies) in RULES[kw]:
        m = re.search(pattern, text, flags=re.IGNORECASE | re.UNICODE)
        if m:
            groups = []
            for g in m.groups():
                g = (g or "").strip()
                groups.append(reflect(g))
            return random.choice(replies), groups
    # Fallback innerhalb der Kategorie
    (pattern, replies) = RULES["default"][0]
    return random.choice(replies), []

def respond(user_input: str) -> str:
    text = _normalize(user_input)
    if not text:
        return random.choice(["Bitte sag etwas mehr.", "Könntest du das genauer ausführen?"])

    kw = _find_keyword(text)
    template, groups = _match_rule(kw, text)
    try:
        return template.format(*groups)
    except Exception:
        return template



---

## Teil 5: Der Chat

Die `chat()`-Funktion startet eine interaktive Sitzung. Tippt `beenden`, um das Gespräch zu verlassen.


In [ ]:
def chat():
    print("=" * 55)
    print("ELIZA -- DOCTOR-Script (Deutsche Version)")
    print("=" * 55)
    print("Tippe 'beenden' zum Beenden.")
    print()
    print("ELIZA: Womit möchtest du beginnen?")
    print()
    while True:
        try:
            user = input("> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nELIZA: Auf Wiedersehen.")
            break
        if not user:
            print("ELIZA: Bitte sag etwas mehr.")
            continue
        if user.lower() in {"beenden"}:
            print("ELIZA: Danke für das Gespräch. Auf Wiedersehen.")
            break
        print(f"ELIZA: {respond(user)}")
        print()


---

## Übung A: Live-Tryout

Führt die nächste Zelle aus, um mit ELIZA zu chatten. Übernehmt dabei am besten eine Rolle mit einer Geschichte, die ihr in einer Therapie erzählen würdet.

**Beobachtungsfragen:**
- Wo wirkt das System *verständig*? An welcher Stelle beginnt ihr, ELIZA Verständnis zuzuschreiben?
- Was kippt ins Artifizielle? Was zerstört die Illusion?
- Wo genau entsteht der **ELIZA-Effekt** — und was löst ihn aus?


In [ ]:
# Startet den Chat -- tippt 'beenden' zum Aufhoeren
chat()


---

## Übung B: ELIZAs Persönlichkeit verändern

**Ziel:** Gleiche Engine, andere Persona — zeigt, wie Daten und Verarbeitungsregeln den Überzeugungseindruck verschieben.

### Was ihr ändern könnt

| Stellschraube | Wo im Code | Beispiel |
|---|---|---|
| **Tonalität** | Antwort-Templates in `RULES` | Von therapeutisch zu bürokratisch, poetisch, genervt |
| **Frageformen** | Antwort-Templates in `RULES` | Statt offener Spiegelung: konfrontative Rückfragen |
| **Default-Antworten** | `RULES["default"]` | Statt *Erzähl weiter* z. B. *Aktenzeichen notiert.* |
| **Prioritäten** | `KEYWORD_ORDER` | Was ELIZA zuerst aufgreift |
| **Neue Keywords** | `KEYWORD_PATTERNS` + `RULES` | Neue Themen, auf die ELIZA reagiert |
| **Pronomen** | `REFLECTIONS` | Du/Sie-Wechsel für formellere Persona |

### Vorschläge für Personas

1. **Sachlich-bürokratisch:** *Ihr Anliegen wurde registriert. Bitte präzisieren Sie.* / *Das ist nicht meine Zuständigkeit.*
2. **Streng-autoritär:** *Das klingt nach Ausreden.* / *Haben Sie dafür Belege?*
3. **Poetisch-assoziativ:** *Trauer… wie ein Fluss, der rückwärts fließt.* / *Was wäre, wenn das Wort 'wenn' nicht existierte?*
4. **Desinteressiert:** *Mhm.* / *Und?* / *Ist mir ehrlich gesagt egal.*
5. **Übertrieben empathisch:** *Oh, das muss SO schwer für dich sein!* / *Ich fühle das TOTAL!*

### Arbeitsschritte

1. Wählt eine Persona
2. Kopiert die Code-Zellen oben und ändert die `RULES` (vor allem die Antwort-Templates und die Default-Antworten)
3. Ändert ggf. die `KEYWORD_ORDER` (Prioritäten) und `REFLECTIONS` (Du/Sie)
4. Startet den Chat und testet

### Diskussionsfragen

- Wie verändert sich der ELIZA-Effekt, wenn die Persona wechselt? Bei welcher Persona entsteht der Eindruck von *Verständnis* am stärksten — und warum?
- Was sagt das über den Zusammenhang von Stimme und Verständnis-Eindruck?
- Welche Persona kommt einem modernen LLM-Chatbot am nächsten? Was hat sich seit 1966 *wirklich* verändert — und was nicht?


In [ ]:
# ============================================================
# PLATZ FUER EURE MODIFIKATIONEN
# ============================================================
# Kopiert die RULES, KEYWORD_ORDER und/oder REFLECTIONS
# von oben hierher und veraendert sie.
# Danach: chat() aufrufen, um die neue Persona zu testen.
# ============================================================

# Beispiel: Nur die Default-Antworten aendern fuer eine
# buerokratische Persona:
#
# RULES["default"] = [
#     (r".*", ["Ihr Vorgang wird bearbeitet. Bitte warten.",
#              "Das ist nicht meine Zustaendigkeit, aber fahren Sie fort.",
#              "Aktenzeichen AZ-2026-ELIZA. Naechster Punkt?",
#              "Bitte fuellen Sie Formular 7b aus.",
#              "Wir kommen hier nicht weiter. Naechstes Thema."])
# ]
#
# chat()


---

## Reflexion

Notiert nach der Übung eure Beobachtungen:

1. **Operationsebene:** ELIZA hat kein Verständnis — nur Pattern-Matching und Substitution. Trotzdem entsteht ein Überzeugungseindruck. Auf welcher Ebene entsteht er? (Sprache? Gesprächsstruktur? Erwartungshaltung?)

2. **Persona & Vibe:** Wie wenig muss sich am Code ändern, damit ELIZA *anders* wirkt? Was sagt das über den Zusammenhang von Stimme, Tonalität und dem Eindruck von Persönlichkeit?

3. **Von ELIZA zu ChatGPT:** Weizenbaum war befremdet, dass Menschen intime Geständnisse an ein Pattern-Matching-System richteten. Heute vertrauen Menschen LLM-Chatbots Ängste, Beziehungsprobleme, Trauer an. Was hat sich verändert — und was wiederholt sich?

4. **Bezug zur spekulativen Kurzgeschichte:** Wenn ihr in eurer Geschichte eine KI-Figur oder ein KI-System habt — auf welcher Ebene erzeugt es seinen Überzeugungseindruck? Und was passiert in der Geschichte, wenn jemand diesen Eindruck für real hält?
